# Text Summarizer — Fine-tuning T5-small on SAMSum

**Pipeline:**
1. Load & inspect the SAMSum dataset
2. Preprocess (clean text)
3. Tokenize for T5
4. Fine-tune with HuggingFace Trainer
5. Save the model
6. Test inference

> **Dataset required:** Place `samsum-train.csv` and `samsum-validation.csv` inside the `ML/data/` folder before running.  
> Download from: https://huggingface.co/datasets/samsum

## 1 — Imports

In [1]:
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,   # ← replaces T5ForConditionalGeneration
    Trainer,
    TrainingArguments,
)
from preprocess import clean_data
from tokenizing import tokenizing_raw_data

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/Users/mr-coder/Desktop/Projects/TextSummarizer/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


## 2 — Load Data

In [2]:
train_data = pd.read_csv('data/samsum-train.csv')
valid_data = pd.read_csv('data/samsum-validation.csv')

print('Train shape :', train_data.shape)
print('Valid shape :', valid_data.shape)
train_data.head()

Train shape : (14731, 3)
Valid shape : (818, 3)


,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\nJ...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\nKim: Bad mood tbh, I was ...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\nSam: i...,"Sam is confused, because he overheard Rick com..."


In [3]:
# Quick sanity-check on first sample
print('--- Dialogue ---')
print(train_data['dialogue'][0])
print()
print('--- Summary ---')
print(train_data['summary'][0])

--- Dialogue ---
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

--- Summary ---
Amanda baked cookies and will bring Jerry some tomorrow.


## 3 — Sample (speeds up training for experimentation)

In [4]:
# Reduce dataset size for faster training.
# Use the full dataset for best quality.
TRAIN_SAMPLES = 4000
VALID_SAMPLES = 500

train_data = train_data.sample(n=TRAIN_SAMPLES, random_state=42).reset_index(drop=True)
valid_data = valid_data.sample(n=VALID_SAMPLES, random_state=42).reset_index(drop=True)

print('Sampled train:', train_data.shape)
print('Sampled valid:', valid_data.shape)

Sampled train: (4000, 3)
Sampled valid: (500, 3)


## 4 — Preprocess

In [5]:
train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary']  = train_data['summary'].apply(clean_data)

valid_data['dialogue'] = valid_data['dialogue'].apply(clean_data)
valid_data['summary']  = valid_data['summary'].apply(clean_data)

print('Preprocessing done.')
print(train_data['dialogue'][0][:200])

Preprocessing done.
violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me 


## 5 — Tokenize

In [6]:
print('Tokenizing training data ...')
train_dataset = train_data.apply(tokenizing_raw_data, axis=1).tolist()

print('Tokenizing validation data ...')
valid_dataset = valid_data.apply(tokenizing_raw_data, axis=1).tolist()

print('Done.')
print('Sample keys:', list(train_dataset[0].keys()))

Tokenizing training data ...
Tokenizing validation data ...
Done.
Sample keys: ['input_ids', 'attention_mask', 'labels']


## 6 — Load Model & Set Device

In [7]:
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print('Device:', device)

MODEL_NAME = "facebook/bart-base"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.to(device)
print('Model loaded:', MODEL_NAME)

Device: mps


/Users/mr-coder/Desktop/Projects/TextSummarizer/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Model loaded: facebook/bart-base


## 7 — Training Arguments

In [8]:
training_args = TrainingArguments(
    output_dir                  = './results',
    num_train_epochs            = 4,
    per_device_train_batch_size = 1,
    per_device_eval_batch_size  = 1,
    gradient_accumulation_steps = 4,
    bf16                        = True,    
    warmup_steps                = 300,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    logging_dir                 = './results/logs',
    logging_steps               = 100,
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = valid_dataset,
)

print('Trainer ready.')

Trainer ready.


## 8 — Train

In [9]:
print('Starting training ...')
trainer.train()
print('Training complete.')

Starting training ...


  0%|          | 0/4000 [00:00<?, ?it/s]

/Users/mr-coder/Desktop/Projects/TextSummarizer/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'loss': 2.6685, 'grad_norm': 16.2005615234375, 'learning_rate': 1.6666666666666667e-05, 'epoch': 0.1}
{'loss': 2.0561, 'grad_norm': 11.319975852966309, 'learning_rate': 3.3333333333333335e-05, 'epoch': 0.2}
{'loss': 1.9732, 'grad_norm': 13.164155006408691, 'learning_rate': 5e-05, 'epoch': 0.3}
{'loss': 1.9271, 'grad_norm': 12.89873218536377, 'learning_rate': 4.8648648648648654e-05, 'epoch': 0.4}
{'loss': 1.8416, 'grad_norm': 6.917459964752197, 'learning_rate': 4.72972972972973e-05, 'epoch': 0.5}
{'loss': 1.9377, 'grad_norm': 11.284689903259277, 'learning_rate': 4.594594594594595e-05, 'epoch': 0.6}
{'loss': 1.8171, 'grad_norm': 9.802279472351074, 'learning_rate': 4.4594594594594596e-05, 'epoch': 0.7}
{'loss': 1.8051, 'grad_norm': 12.823324203491211, 'learning_rate': 4.324324324324325e-05, 'epoch': 0.8}
{'loss': 1.7864, 'grad_norm': 10.095512390136719, 'learning_rate': 4.189189189189189e-05, 'epoch': 0.9}
{'loss': 1.7083, 'grad_norm': 11.687397956848145, 'learning_rate': 4.0540540540540

  0%|          | 0/500 [00:00<?, ?it/s]

...

  0%|          | 0/500 [00:00<?, ?it/s]

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}


{'eval_loss': 1.502842903137207, 'eval_runtime': 41.3214, 'eval_samples_per_second': 12.1, 'eval_steps_per_second': 12.1, 'epoch': 4.0}


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


{'train_runtime': 4774.8579, 'train_samples_per_second': 3.351, 'train_steps_per_second': 0.838, 'train_loss': 1.3795615119934082, 'epoch': 4.0}
Training complete.


## 9 — Save Model

In [10]:
SAVE_PATH = './saved_summary_model'

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f'Model saved to {SAVE_PATH}')

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}


Model saved to ./saved_summary_model


## 10 — Test Inference

In [13]:
def summarize(dialogue: str) -> str:
    import re
    text = re.sub(r'\r\n|\r|\n', ' ', dialogue)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()          # ← no .lower() and no "summarize: " prefix

    inputs = tokenizer(
        text,
        padding="max_length",
        max_length=1024,         # ← updated to 1024
        truncation=True,
        return_tensors='pt',
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=512,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
            length_penalty=2.0,   # ← BART benefits from this
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)